In [ ]:
import pandas as pd

df = pd.read_excel(r"/kaggle/working/datasets-dgtx/DATA_DG.xlsx")   # hoặc path đúng của bạn

assert {"content", "summary", "grade"}.issubset(df.columns)
df = df.dropna().reset_index(drop=True)


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "VietAI/vit5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer.model_max_length)  # ~512 tokens

from datasets import Dataset

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 256

def preprocess(batch):
    prompts = [
        f"Tóm tắt văn bản cho học sinh lớp {g}: {c}"
        for c, g in zip(batch["content"], batch["grade"])
    ]

    model_inputs = tokenizer(
        prompts,
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT_LEN,
    )

    labels = tokenizer(
        batch["summary"],
        truncation=True,
        max_length=MAX_TARGET_LEN,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

tokenized_ds = dataset.map(
    preprocess,
    batched=True,   # ⭐ BẮT BUỘC
    remove_columns=dataset["train"].column_names
)

print(tokenized_ds["train"].column_names)

In [ ]:
# ============================================
# CELL 1: OPTUNA HYPERPARAMETER TUNING
# ============================================

import optuna
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = 1
    epochs = trial.suggest_int("num_train_epochs", 3, 8)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    torch.cuda.empty_cache()
    import gc
    gc.collect()

    args = TrainingArguments(
        output_dir=f"./optuna_trial_{trial.number}",
        eval_strategy="no",
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="no",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        label_smoothing_factor=0.1,
        max_grad_norm=1.0,
        fp16=True,
        gradient_checkpointing=True,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        ddp_find_unused_parameters=False,
        report_to="none"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=None
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=None,
    )

    trainer.train()
    train_loss = trainer.state.log_history[-1].get("train_loss", float('inf'))
    
    del model
    del trainer
    del data_collator
    del args
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    
    return train_loss

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

best_params = study.best_params
print("=" * 50)
print("BEST HYPERPARAMETERS:")
print("=" * 50)
for key, value in best_params.items():
    print(f"{key}: {value}")
print("=" * 50)

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    import numpy as np
    
    # preds từ Trainer là logits, cần lấy argmax để có token IDs
    if isinstance(preds, tuple):
        preds = preds[0]
    
    if isinstance(preds, np.ndarray):
        # Lấy token ID có xác suất cao nhất (argmax)
        if preds.ndim == 3:  # (batch_size, seq_len, vocab_size)
            preds = np.argmax(preds, axis=-1)
        elif preds.ndim > 3:
            preds = preds.reshape(preds.shape[0], -1)
            preds = np.argmax(preds, axis=-1)
    
    # Xử lý labels - loại bỏ padding tokens (-100)
    if isinstance(labels, np.ndarray):
        labels = labels.tolist()
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Decode labels - loại bỏ -100 trước khi decode
    decoded_labels = []
    for label in labels:
        if isinstance(label, np.ndarray):
            label = label.tolist()
        # Loại bỏ -100 (ignore index)
        valid_label = [l for l in label if l != -100]
        decoded_labels.append(tokenizer.decode(valid_label, skip_special_tokens=True))

    # Đảm bảo decoded_preds và decoded_labels là list các string
    decoded_preds = [str(p) if p else "" for p in decoded_preds]
    decoded_labels = [str(l) if l else "" for l in decoded_labels]

    # -------- ROUGE --------
    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # -------- BERTScore --------
    bert_result = bertscore.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang="vi",
        model_type="bert-base-multilingual-cased"
    )

    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bertscore_precision": sum(bert_result["precision"]) / len(bert_result["precision"]),
        "bertscore_recall": sum(bert_result["recall"]) / len(bert_result["recall"]),
        "bertscore_f1": sum(bert_result["f1"]) / len(bert_result["f1"]),
    }
    
import pandas as pd

def save_metrics(trainer, filename="metrics.csv"):
    logs = trainer.state.log_history
    df_metrics = pd.DataFrame(logs)
    df_metrics.to_csv(filename, index=False)
    return df_metrics

In [ ]:
# ============================================
# CELL 2: FINAL TRAINING VỚI BEST PARAMETERS
# ============================================

from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
import torch
import os

torch.cuda.empty_cache()

# Kiểm tra xem có checkpoint để resume không
resume_from_checkpoint = None
checkpoint_dir = "./vit5_final"
if os.path.exists(checkpoint_dir):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint-")]
    if checkpoints:
        # Lấy checkpoint mới nhất
        checkpoint_numbers = [int(c.split("-")[1]) for c in checkpoints]
        latest_checkpoint = f"checkpoint-{max(checkpoint_numbers)}"
        resume_from_checkpoint = os.path.join(checkpoint_dir, latest_checkpoint)
        print(f"Tìm thấy checkpoint: {resume_from_checkpoint}")
        print("Sẽ tiếp tục training từ checkpoint này...")
    else:
        print("Không tìm thấy checkpoint, bắt đầu training từ đầu")
else:
    print("Không tìm thấy thư mục checkpoint, bắt đầu training từ đầu")

best_learning_rate = 0.0003290826760268271
best_num_train_epochs = 5
best_weight_decay = 0.004878586459798251

final_args = TrainingArguments(
    output_dir="./vit5_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=best_learning_rate,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=best_num_train_epochs,
    weight_decay=best_weight_decay,
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,
    save_total_limit=5,
    report_to="none"
)

# Load model từ checkpoint nếu có, nếu không thì từ pretrained
if resume_from_checkpoint:
    final_model = AutoModelForSeq2SeqLM.from_pretrained(resume_from_checkpoint)
else:
    final_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

final_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=final_model,
    padding=True,
    pad_to_multiple_of=None
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=final_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("=" * 50)
print("BẮT ĐẦU TRAINING VỚI BEST PARAMETERS")
if resume_from_checkpoint:
    print(f"Resume từ checkpoint: {resume_from_checkpoint}")
print("=" * 50)

if resume_from_checkpoint:
    final_trainer.train(resume_from_checkpoint=resume_from_checkpoint)
else:
    final_trainer.train()

final_trainer.save_model("./vit5_grade_summary")
tokenizer.save_pretrained("./vit5_grade_summary")

print("=" * 50)
print("TRAINING HOÀN TẤT!")
print("Model đã được lưu tại: ./vit5_grade_summary")
print("=" * 50)

metrics_df = save_metrics(final_trainer, "vit5_metrics.csv")
print("\nMetrics:")
print(metrics_df.head())
print("\nColumns:", metrics_df.columns.tolist())

import matplotlib.pyplot as plt

eval_df = metrics_df[metrics_df["eval_rouge1"].notna()].copy()
if "epoch" not in eval_df.columns or eval_df["epoch"].isna().all():
    if "step" in eval_df.columns:
        steps_per_epoch = len(tokenized_ds["train"]) // (final_args.per_device_train_batch_size * final_args.gradient_accumulation_steps)
        eval_df["epoch"] = eval_df["step"] / steps_per_epoch
    else:
        eval_df["epoch"] = range(len(eval_df))

if len(eval_df) > 0:
    plt.figure(figsize=(10,6))
    if "eval_rouge1" in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df["eval_rouge1"], label="ROUGE-1")
    if "eval_rougeL" in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df["eval_rougeL"], label="ROUGE-L")
    bertscore_col = "eval_bertscore_f1" if "eval_bertscore_f1" in eval_df.columns else "bertscore_f1"
    if bertscore_col in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df[bertscore_col], label="BERTScore-F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Evaluation Metrics over Epochs")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Warning: No evaluation data found for plotting")

In [2]:
# ============================================
# CELL: TEST MODEL
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

# Đường dẫn đến model đã train
MODEL_PATH = "../../Model_DG/vit5_grade_summary"
ORIGINAL_MODEL = "VietAI/vit5-base"

# Kiểm tra xem model có tồn tại không
if not os.path.exists(MODEL_PATH):
    print(f"Model không tồn tại tại: {MODEL_PATH}")
    print("Vui lòng train model trước hoặc kiểm tra đường dẫn")
else:
    print(f"Đang load model từ: {MODEL_PATH}")
    
    # Load tokenizer - thử từ checkpoint trước, nếu lỗi thì dùng model gốc
    # (Tokenizer không thay đổi sau training, chỉ model weights thay đổi)
    try:
        print("Đang load tokenizer từ checkpoint...")
        test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
        print("✓ Tokenizer loaded từ checkpoint (slow tokenizer)")
    except Exception as e1:
        try:
            print(f"⚠ Không thể load slow tokenizer: {str(e1)[:100]}")
            print("Đang thử load fast tokenizer...")
            test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
            print("✓ Tokenizer loaded từ checkpoint (fast tokenizer)")
        except Exception as e2:
            print(f"⚠ Không thể load tokenizer từ checkpoint: {str(e2)[:100]}")
            print(f"⚠ File tokenizer.json có thể bị corrupt")
            print(f"Đang load tokenizer từ model gốc {ORIGINAL_MODEL}...")
            print("(Lưu ý: Tokenizer không thay đổi sau training, nên dùng model gốc vẫn đúng)")
            test_tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL)
            print("✓ Tokenizer loaded từ model gốc")
    
    # Load model weights từ checkpoint (đây là phần quan trọng - model đã được fine-tune)
    print("\nĐang load model weights từ checkpoint...")
    test_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
    print("✓ Model weights loaded từ checkpoint")
    
    # Chuyển model sang GPU nếu có
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_model.to(device)
    test_model.eval()
    
    print(f"Model đã được load vào: {device}")
    print("=" * 50)

def calculate_relevance_score(summary_text, original_text):
    """
    Tính điểm độ liên quan giữa tóm tắt và văn bản gốc dựa trên keyword overlap
    
    Returns:
        float: Điểm từ 0.0 đến 1.0 (càng cao càng liên quan)
    """
    import re
    
    stopwords = {'là', 'của', 'và', 'với', 'cho', 'để', 'trong', 'trên', 'từ', 'đến', 'một', 'các', 'những', 'này', 'đó', 'đã', 'được', 'sẽ', 'có', 'không', 'theo', 'sau', 'trước', 'khi', 'nếu', 'thì', 'mà', 'nhưng', 'hoặc'}
    
    def extract_keywords(text):
        text = re.sub(r'[^\w\s]', ' ', text.lower())
        words = text.split()
        keywords = [w for w in words if len(w) > 2 and w not in stopwords]
        return set(keywords)
    
    summary_keywords = extract_keywords(summary_text)
    original_keywords = extract_keywords(original_text)
    
    if len(original_keywords) == 0:
        return 1.0
    
    intersection = len(summary_keywords & original_keywords)
    union = len(summary_keywords | original_keywords)
    
    if union == 0:
        return 0.0
    
    jaccard = intersection / union
    coverage = intersection / len(original_keywords) if len(original_keywords) > 0 else 0.0
    relevance = (jaccard * 0.4 + coverage * 0.6)
    
    return relevance


def summarize(text, grade, max_new_tokens=None, use_sampling=True, temperature=0.9, debug=False):
    """
    Tóm tắt văn bản cho học sinh theo lớp
    
    Args:
        text: Văn bản cần tóm tắt
        grade: Lớp học (1-5)
        max_new_tokens: Số token tối đa (nếu None, tự động tính dựa trên 50% số câu)
        use_sampling: Không dùng nữa (luôn dùng sampling để tạo đa dạng)
        temperature: Độ đa dạng của output (0.1-1.5)
            - 0.7-0.9: Đa dạng vừa phải, cân bằng giữa chất lượng và đa dạng (khuyến nghị)
            - 0.5-0.7: Ít đa dạng hơn, ổn định hơn
            - 0.9-1.2: Đa dạng cao, mỗi lần chạy khác nhau nhiều
            - >1.2: Rất đa dạng nhưng có thể kém chính xác
        debug: Nếu True, in thông tin debug
    
    Returns:
        Tóm tắt văn bản (đã được cắt ở câu hoàn chỉnh và loại bỏ phần không liên quan)
        Mỗi lần chạy sẽ tạo ra tóm tắt khác nhau do sử dụng sampling
    """
    import re
    
    if grade < 1:
        grade = 1
    if grade > 5:
        grade = 5
    
    # Tính số câu trong văn bản gốc
    original_sentences = re.split(r'[.!?]+\s*', text)
    original_sentences = [s.strip() for s in original_sentences if s.strip()]
    num_original_sentences = len(original_sentences)
    
    # Tính max_new_tokens dựa trên 50% số câu
    if max_new_tokens is None:
        avg_tokens_per_sentence = {
            1: 15,
            2: 18,
            3: 22,
            4: 25,
            5: 30  # Tăng lên để model generate dài hơn
        }.get(grade, 25)
        
        target_sentences = max(1, int(num_original_sentences * 0.5))
        max_new_tokens = target_sentences * avg_tokens_per_sentence
        
        # Tăng giới hạn đáng kể để ép model generate dài hơn (vì model được train với MAX_TARGET_LEN=64)
        # Với 21 câu gốc, 50% = ~10-11 câu, mỗi câu 30 tokens = 300-330 tokens
        min_limit = {1: 120, 2: 150, 3: 200, 4: 250, 5: 350}.get(grade, 300)
        max_limit = {1: 300, 2: 400, 3: 500, 4: 650, 5: 800}.get(grade, 700)
        max_new_tokens = max(min_limit, min(max_new_tokens, max_limit))
    
    # Tăng min_new_tokens lên 50% để đảm bảo tóm tắt đủ dài
    min_new_tokens = int(max_new_tokens * 0.5)
    
    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {text}"
    
    inputs = test_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)
    
    with torch.no_grad():
        # Dùng sampling với temperature để tạo sự đa dạng mỗi lần chạy
        # Temperature cao hơn = đa dạng hơn, nhưng có thể kém chính xác hơn
        output = test_model.generate(
            **inputs,
            min_new_tokens=min_new_tokens,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # Bật sampling để tạo đa dạng
            temperature=temperature,  # Temperature từ parameter (mặc định 0.9)
            top_p=0.92,  # Nucleus sampling - chỉ xét top 92% tokens
            top_k=50,  # Top-k sampling - chỉ xét top 50 tokens
            length_penalty=2.0,  # Khuyến khích generate dài
            repetition_penalty=1.2,  # Giảm lặp lại
            no_repeat_ngram_size=2,  # Tránh lặp cụm từ
            eos_token_id=test_tokenizer.eos_token_id,
            pad_token_id=test_tokenizer.pad_token_id
        )
    
    summary_raw = test_tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Debug: Tính toán thông tin về summary gốc
    original_summary_length = len(summary_raw.split())
    # Tính số tokens đã generate (loại bỏ input tokens)
    input_length = inputs['input_ids'].shape[1]
    output_length = output[0].shape[0]
    num_generated_tokens = output_length - input_length
    num_original_sentences_in_summary = len(re.findall(r'[.!?]', summary_raw))
    
    summary = summary_raw
    
    # Post-processing: Cắt ở câu hoàn chỉnh và loại bỏ phần không liên quan
    sentences = re.split(r'([.!?]+\s*)', summary)
    
    complete_sentences = []
    for i in range(0, len(sentences) - 1, 2):
        if i + 1 < len(sentences):
            sentence = (sentences[i] + sentences[i + 1]).strip()
            if sentence:
                complete_sentences.append(sentence)
    
    # Xử lý câu cuối - CHỈ giữ nếu có dấu kết thúc câu (để tránh cắt giữa câu)
    if len(sentences) % 2 == 1 and sentences[-1].strip():
        last_sentence = sentences[-1].strip()
        # CHỈ giữ nếu có dấu kết thúc câu (hoàn chỉnh)
        if re.search(r'[.!?]$', last_sentence):
            complete_sentences.append(last_sentence)
        # Nếu không có dấu kết thúc, bỏ qua (câu bị cắt)
    
    # Loại bỏ câu không liên quan hoặc lặp lại
    filtered_sentences = []
    seen_content = set()
    min_relevance_threshold = 0.05  # Giảm xuống để giữ nhiều câu hơn
    
    for sentence in complete_sentences:
        sentence_relevance = calculate_relevance_score(sentence, text)
        
        # Chỉ lọc câu có độ liên quan RẤT thấp (< 0.03) - rất lỏng để giữ nhiều nội dung
        if sentence_relevance < 0.03:
            continue
        
        normalized = re.sub(r'\s+', ' ', sentence.lower().strip())
        
        # Kiểm tra duplicate (tăng ngưỡng lên 0.9 để linh hoạt hơn, chỉ lọc câu gần như giống hệt)
        is_duplicate = False
        for seen in seen_content:
            if len(normalized) > 0 and len(seen) > 0:
                similarity = len(set(normalized.split()) & set(seen.split())) / max(len(normalized.split()), len(seen.split()))
                if similarity > 0.9:  # Tăng lên 0.9 để chỉ lọc câu gần như giống hệt
                    is_duplicate = True
                    break
        
        if not is_duplicate:
            filtered_sentences.append(sentence)
            seen_content.add(normalized)
    
    # Xử lý kết quả cuối cùng - GIỮ TẤT CẢ các câu hợp lệ, không dừng sớm
    if filtered_sentences:
        # Chỉ loại bỏ câu có độ liên quan RẤT thấp ở cuối (nếu có nhiều câu)
        # Nhưng vẫn giữ tất cả các câu hợp lệ
        final_summary = []
        for i, sentence in enumerate(filtered_sentences):
            sentence_relevance = calculate_relevance_score(sentence, text)
            
            # Chỉ bỏ qua câu có độ liên quan RẤT thấp (< 0.03) và chỉ khi đã có ít nhất 5 câu
            # Điều này đảm bảo giữ được nhiều nội dung
            if sentence_relevance < 0.03 and len(final_summary) >= 5:
                # Kiểm tra xem câu tiếp theo có tốt hơn không
                if i + 1 < len(filtered_sentences):
                    next_relevance = calculate_relevance_score(filtered_sentences[i+1], text)
                    if next_relevance < 0.03:
                        # Nếu câu tiếp theo cũng rất thấp, dừng ở đây
                        break
                else:
                    # Nếu đây là câu cuối, vẫn giữ nếu đã có ít nhất 5 câu
                    break
            
            final_summary.append(sentence)
        
        summary = ' '.join(final_summary)
    else:
        # Fallback: lấy các câu hoàn chỉnh từ summary gốc (có dấu kết thúc)
        sentences_original = re.split(r'([.!?]+\s*)', summary)
        fallback_sentences = []
        for i in range(0, len(sentences_original) - 1, 2):
            if i + 1 < len(sentences_original):
                sentence = (sentences_original[i] + sentences_original[i + 1]).strip()
                if sentence:
                    fallback_sentences.append(sentence)
        
        if fallback_sentences:
            summary = ' '.join(fallback_sentences)
        else:
            summary = summary.strip()
    
    # Debug output
    if debug:
        print(f"\n[DEBUG] Số câu gốc: {num_original_sentences}")
        print(f"[DEBUG] max_new_tokens: {max_new_tokens}, min_new_tokens: {min_new_tokens}")
        print(f"[DEBUG] Số tokens đã generate: {num_generated_tokens}")
        print(f"[DEBUG] Số từ trong summary gốc: {original_summary_length}")
        print(f"[DEBUG] Số câu trong summary gốc: {num_original_sentences_in_summary}")
        print(f"[DEBUG] Số câu sau post-processing: {len(final_summary) if filtered_sentences else len(fallback_sentences) if fallback_sentences else 0}")
    
    return summary

# ============================================
# TEST VỚI CÁC VÍ DỤ
# ============================================

print("\n" + "=" * 50)
print("TEST MODEL")
print("=" * 50)
test_text_3 = """TÌNH HÌNH XÃ HỘI GIAO CHÂU
Về mặt xã hội, chính sách bóc lột nặng nề của triều đình nhà Đường và sự tham lam vơ vét cúa bọn quan lại đô hộ đã nhanh chóng đẩy người dân Giao Châu vào con đường bần cùng hóa. 
Sự phân hóa giai cấp trong xã hội ngày càng sâu sắc. Đặc biệt, từ nửa sau thế kỳ VIII, lụt lội và hạn hán xảy ra thường xuyên, chiến tranh liên tiếp do các nước láng giềng là Hoàn 
Vương (Champa) và Nam Chiếu gây ra, khiến sức sản xuất bị phá hoại, đời sống nhân dân ngày càng thêm cơ cực. về mặt văn hoá, nhà Đường đă đu nhập đạo Nho, đạo Phật và đạo Giáo với 
mục đích dựa vào sự phát triển văn hóa để nô dịch nhân dân Giao Châu. Nho giáo thời Đường ở Giao Châu chưa thể xem là phát triển, song cũng được truyền bá sâu rộng trong tầng lớp 
trên của xã hội. Trường học dạy chữ Hán được mờ nhiều ờ các phủ, châu. Trong tầng lớp Hào trưởng người Việt, một số gia đình đã cho con em học hành. Họ được tham gia thi cử và đỗ 
đạt ở Bắc triều, một số người được tuyển dụng vào bộ máy của chính quyền đô hộ, như trường hợp anh em Khương Công Phụ và Khương Công Phục. Tuy nhiên, việc học và thi cử ở Giao Châu 
vẫn bị hạn chế. Lệ nhà Đường năm Hội Xương thứ 5 (845) quy định: "An Nam đưa vào thi Tiến sĩ không được quá tám người, Minh kinh không được quá mười người". Dưới triều Đường, đạo 
Phật được truyền bá vào Giao Châu với hai phái Thiền tông của Phật giáo Trung Quốc. Phái thứ nhất do Tì Ni Đa Lưu Chi sáng lập, truyền bá vào Giao Châu cuối thế kỷ VI, trung tâm là 
chùa Pháp Vân (Thuận Thành, Bắc Ninh). Phái thứ hai do Vô Ngôn Thông sáng lập, truyền vào Giao Châu đầu thế kỷ IX, trung tâm là chùa Kiến Sơ (Phù Đổng, ngoại thành Hà Nội). Bên 
cạnh đạo Phật, Đạo giáo cũng khá phát triển và chịu ảnh hưởng sâu sắc của Trung Quốc. Nhà Đường đã cho nhiều đạo sĩ, phù thủy sang Giao Châu, trong đó có Tiết độ sứ Cao Biền với 
những phương thức tà ma, bùa chú để yểm trừ "long mạch"... Tuy nhiên, sự phát triển của đạo Nho, đạo Phật và Đạo giáo đều dung hòa được với tín ngưỡng dân gian cổ truyền của người 
Việt (như tục thờ thần sông, thần núi, thờ các vị anh hùng dân tộc.. tạo nên sức mạnh của dân tộc Việt, chống lại sự nô dịch văn hoá của ngoại bang."""
test_grade_3 = 5

print(f"\n" + "-" * 50)
print(f"[TEST 3] Lớp {test_grade_3}")
print(f"Văn bản gốc: {test_text_3}")
print(f"\nTóm tắt:")
summary_3 = summarize(test_text_3, test_grade_3, debug=True)
print(summary_3)

print("\n" + "=" * 50)
print("HOÀN TẤT TEST!")
print("=" * 50)

Đang load model từ: ../../Model_DG/vit5_grade_summary
Đang load tokenizer từ checkpoint...
✓ Tokenizer loaded từ checkpoint (slow tokenizer)

Đang load model weights từ checkpoint...
✓ Model weights loaded từ checkpoint
Model đã được load vào: cuda

TEST MODEL

--------------------------------------------------
[TEST 3] Lớp 5
Văn bản gốc: TÌNH HÌNH XÃ HỘI GIAO CHÂU
Về mặt xã hội, chính sách bóc lột nặng nề của triều đình nhà Đường và sự tham lam vơ vét cúa bọn quan lại đô hộ đã nhanh chóng đẩy người dân Giao Châu vào con đường bần cùng hóa. 
Sự phân hóa giai cấp trong xã hội ngày càng sâu sắc. Đặc biệt, từ nửa sau thế kỳ VIII, lụt lội và hạn hán xảy ra thường xuyên, chiến tranh liên tiếp do các nước láng giềng là Hoàn 
Vương (Champa) và Nam Chiếu gây ra, khiến sức sản xuất bị phá hoại, đời sống nhân dân ngày càng thêm cơ cực. về mặt văn hoá, nhà Đường đă đu nhập đạo Nho, đạo Phật và đạo Giáo với 
mục đích dựa vào sự phát triển văn hóa để nô dịch nhân dân Giao Châu. Nho giáo thời Đường 

c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `num_beams` is set to 1. However, `length_penalty` is set to `2.0` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `length_penalty`.
  warnings.warn(



[DEBUG] Số câu gốc: 17
[DEBUG] max_new_tokens: 350, min_new_tokens: 175
[DEBUG] Số tokens đã generate: -230
[DEBUG] Số từ trong summary gốc: 258
[DEBUG] Số câu trong summary gốc: 7
[DEBUG] Số câu sau post-processing: 7
Ngày nay, chính sách bóc lột nặng nề của triều đình nhà Đường và sự tham lam vơ vét đã đẩy người dân Giao Châu vào con đường bần cùng. Từ nửa sau thế kỳ VIII, lụt lội và hạn hán kéo dài làm đời sống nhân dân càng thêm khó khăn, đặc biệt từ cuối thời kỳ VII đến cuối Thế kỳ IX, úng ngập xảy ra thường xuyên khiến sản xuất bị phá hủy hoặc cuộc chiến tiếp diễn. Nho giáo được truyền bá qua hai phái Thiền tông, với hai học trò là Tì Ni Đa Lưu Chi và Vô Ngôn Thông. Đạo Phật cũng có ảnh hưởng tích cực của Trung Quốc, như việc cho đạo sĩ thi cử và đỗ đạt ở Bắc triều. Tuy nhiên, việc học tập ở Giao Chau vẫn bị hạn chế khi Lệ nhà Đàng quy định rằng an Nam đưa tiến sĩ không quá tám người và Minh kinh chỉ khoảng mười người. Khổng giáo còn phát triển khá sâu sắc trong tầng lớp Hào trư

In [7]:
# ============================================
# LOAD MODEL ĐÃ TRAIN
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = "../../Model_DG_ver2/vit5_grade_summary"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# ============================================
# HÀM TÓM TẮT 1 VĂN BẢN
# ============================================

def summarize_text(content, grade,
                   max_input_len=768,
                   max_target_len=256,
                   mode="sample",
                   length_option="medium"):  # beam | sample

    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {content}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    ).to(device)

    with torch.no_grad():

        if mode == "beam":
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_target_len,
                num_beams=5,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )

        elif mode == "sample":
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_target_len,
                do_sample=True,
                top_k=50,
                top_p=0.9,
                temperature=0.8,
                no_repeat_ngram_size=3,
            )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary


# ============================================
# TEST VỚI 1 VĂN BẢN BẤT KỲ
# ============================================

text = """
Chị là người con gái đẹp.Chị được học ít và thay mẹ chăm sóc em.Mỗi lần ra khỏi nhà mẹ đều dặn đi dặn lại là không được mỡ cửa vì biết có nhiều chàng trai ngắm nghía chị.Các em lập gia đình,chỉ còn mẹ và chị.Mẹ thương chị ngày nào cũng mỡ cửa nhìn ra ngõ.Mỗi lần như thế chị lại ra đóng cữa và dặn mẹ:”Mẹ đừng mỡ cửa nhé,gió làm mẹ lạnh,con lo cho mẹ lắm!” Mẹ rơi nuớc mắt."""

grade = 5

result = summarize_text(text, grade)
print("Văn bản: ", text)
print("Tóm tắt: ", result)

Văn bản:  
Chị là người con gái đẹp.Chị được học ít và thay mẹ chăm sóc em.Mỗi lần ra khỏi nhà mẹ đều dặn đi dặn lại là không được mỡ cửa vì biết có nhiều chàng trai ngắm nghía chị.Các em lập gia đình,chỉ còn mẹ và chị.Mẹ thương chị ngày nào cũng mỡ cửa nhìn ra ngõ.Mỗi lần như thế chị lại ra đóng cữa và dặn mẹ:”Mẹ đừng mỡ cửa nhé,gió làm mẹ lạnh,con lo cho mẹ lắm!” Mẹ rơi nuớc mắt.
Tóm tắt:  Chị là một cô gái xinh đẹp, được học ít và chăm sóc em, nhưng luôn dặn mẹ không được mỡ cửa vì có nhiều chàng trai ngắm nghía chị. Khi các em lập gia đình, chỉ còn mẹ và chị, mẹ rất thương chị và luôn lo lắng cho em. Mỗi lần ra ngoài, chị lại đóng cữa và dặn mẹ đừng mỡ cửa, vì có thể làm mẹ lạnh và lo cho mẹ. Mẹ rơi nước mắt vì tình thương của chị dành cho con.


Níc Vu-gic, một diễn giả truyền cảm hứng người Úc, sinh năm 1982 và chỉ có một bàn chân với hai ngón chân nhỏ. Do khác biệt về ngoại hình, Níc đã học cách dùng chân và một cái cán để viết chữ, đánh bàn phím máy vi tính và chăm sóc bản thân. Anh đã vượt qua khó khăn để biến bản thân thành điều kì diệu trong cuộc sống. Cuối cùng, Ní đã biến bản Thân trở thành một biểu tượng của nghị lực sống và lan toả năng lượng tích cực tới người xung quanh.

Níc Vu-gic là một diễn giả truyền cảm hứng người Úc, sinh năm 1982 và chỉ có một bàn chân nhỏ. Sự khác biệt về ngoại hình khiến anh từng bị bạn bè chê cười. Trong thời gian khó khăn, Níc phải chấp nhận sống chung với sự thiếu sót trên cơ thể mình và học cách dùng chân và cán để viết chữ và đánh bàn phím. Anh đã vượt qua khó khăn để biến mình thành điều kì diệu trong cuộc sống.

Níc Vu-gic là một diễn giả truyền cảm hứng người Úc, sinh năm 1982 và chỉ có một bàn chân với hai ngón chân nhỏ. Sự khác biệt về ngoại hình khiến Níc từng bị bạn bè chê cười và trêu chọc. Trong thời gian khó khăn, Níc đã học cách dùng chân và một cái cán để viết chữ, đánh bàn phím máy vi tính và chăm sóc bản thân. Anh đã vượt qua khó khăn và biến bản thân thành điều kì diệu trong cuộc sống.

In [9]:
# ============================================
# LOAD MODEL
# ============================================
import re
import unicodedata
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from vncorenlp import VnCoreNLP

MODEL_PATH = "../../Model_DG_ver2/vit5_grade_summary"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

model.to(device)
model.eval()


# ============================================
# LOAD VNCORENLP
# ============================================

vncore = VnCoreNLP(
    "../../../VnCoreNLP-master/VnCoreNLP-1.1.1.jar",
    annotators="wseg,pos,ner",
    max_heap_size='-Xmx2g'
)


# ============================================
# ĐẾM SỐ TỪ
# ============================================

def count_words(text):

    sentences = vncore.tokenize(text)

    words = []
    for sent in sentences:
        words.extend(sent)

    return len(words)


# ============================================
# WORD → TOKEN CONVERSION
# ============================================

def word_to_token_estimate(word_count):
    """
    Với tiếng Việt:
    1 word ≈ 1.3 token (thực nghiệm với tokenizer T5)
    """
    return int(word_count * 1.3)


# ============================================
# CẮT SUMMARY Ở CÂU HOÀN CHỈNH
# ============================================

def clean_summary(summary):

    last_period = summary.rfind(".")

    if last_period != -1:
        summary = summary[:last_period + 1]

    return summary.strip()


def clean_vietnamese_text(text):
    
    # 1 normalize unicode
    text = unicodedata.normalize("NFC", text)

    # 2 fix double spaces
    text = re.sub(r"\s+", " ", text)

    # 3 fix dính chữ hoa
    text = re.sub(r"([a-zàáạảãăắằẳẵặâấầẩẫậ])([A-Z])", r"\1 \2", text)

    # 4 fix chữ viết hoa toàn bộ
    def fix_upper(word):
        if word.isupper() and len(word) > 2:
            return word.capitalize()
        return word

    text = " ".join([fix_upper(w) for w in text.split()])

    # 5 fix các lỗi phổ biến
    corrections = {
        "Đĩnhchi": "Đĩnh Chi",
        "Mạc Đĩnhchi": "Mạc Đĩnh Chi",
        "Đĩnh chi": "Đĩnh Chi",
        "Đĩnh CHI": "Đĩnh Chi",
    }

    for wrong, correct in corrections.items():
        text = text.replace(wrong, correct)

    return text

import re
import unicodedata


def vietnamese_text_normalization(text, vncore=None):
    """
    Vietnamese Text Normalization Layer
    Works for ANY Vietnamese text
    """

    # ====================================
    # 1. Unicode normalization
    # ====================================
    text = unicodedata.normalize("NFC", text)

    # ====================================
    # 2. Remove duplicated spaces
    # ====================================
    text = re.sub(r"\s+", " ", text)

    # ====================================
    # 3. Fix spacing around punctuation
    # ====================================
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"([,.!?;:])([^\s])", r"\1 \2", text)

    # ====================================
    # 4. Fix glued words
    # Example: Đĩnhchi -> Đĩnh Chi
    # ====================================
    text = re.sub(
        r"([a-zàáạảãăắằẳẵặâấầẩẫậđèéẹẻẽêếềểễệìíịỉĩòóọỏõôốồổỗộơớờởỡợùúụủũưứừửữựỳýỵỷỹ])([A-ZÀÁẠẢÃĂẮẰẲẴẶÂẤẦẨẪẬĐ])",
        r"\1 \2",
        text
    )

    # ====================================
    # 5. Fix ALL CAPS words
    # ====================================
    words = text.split()

    fixed_words = []
    for w in words:
        if w.isupper() and len(w) > 2:
            fixed_words.append(w.capitalize())
        else:
            fixed_words.append(w)

    text = " ".join(fixed_words)

    # ====================================
    # 6. Capitalize sentences
    # ====================================
    sentences = re.split(r'(?<=[.!?]) +', text)

    def capitalize_first_letter(sentence):
        if not sentence:
            return sentence
        return sentence[0].upper() + sentence[1:]

    sentences = [capitalize_first_letter(s) for s in sentences]

    text = " ".join(sentences)

    # ====================================
    # 7. Use NER to fix proper names
    # ====================================
    if vncore is not None:

        annotated = vncore.annotate(text)

        entities = set()

        for sent in annotated['sentences']:
            for token in sent:
                if token['nerLabel'] != 'O':
                    entities.add(token['form'])

        for ent in entities:
            pattern = re.compile(ent, re.IGNORECASE)
            text = pattern.sub(ent, text)

    return text.strip()

# ============================================
# HÀM TÓM TẮT
# ============================================

def summarize_text(
        content,
        grade,
        length_option="medium",
        max_input_len=768,
        mode="sample"
):

    # -------------------------------
    # 1. ĐẾM SỐ TỪ VĂN BẢN
    # -------------------------------

    total_words = count_words(content)

    # -------------------------------
    # 2. TÍNH ĐỘ DÀI SUMMARY
    # -------------------------------

    if length_option == "short":
        summary_words = int(total_words * 0.35)

    elif length_option == "medium":
        summary_words = int(total_words * 0.50)

    elif length_option == "long":
        summary_words = int(total_words * 0.75)

    else:
        summary_words = int(total_words * 0.50)

    # -------------------------------
    # 3. WORD → TOKEN
    # -------------------------------

    max_len = word_to_token_estimate(summary_words)
    max_len = min(max_len, 256)
    min_len = max(1, min(int(max_len * 0.3), max_len))

    # -------------------------------
    # 4. TOKENIZE INPUT
    # -------------------------------

    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {content}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    ).to(device)

    # -------------------------------
    # 5. GENERATE (min/max_new_tokens ghi đè generation_config.max_new_tokens=256)
    # -------------------------------

    with torch.no_grad():

        if mode == "beam":

            output_ids = model.generate(
                **inputs,
                min_new_tokens=min_len,
                max_new_tokens=max_len,
                num_beams=5,
                length_penalty=1.0,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )

        else:

            output_ids = model.generate(
    **inputs,
    min_new_tokens=60,
    max_new_tokens=200,
    do_sample=True,
    top_k=40,
    top_p=0.92,
    temperature=0.9,
    repetition_penalty=1.2,
)

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )
    summary = vietnamese_text_normalization(summary, vncore)
    summary = clean_summary(summary)

    return summary, total_words, summary_words


# ============================================
# TEST
# ============================================

text = """Ngày xửa ngay xưa, xưa lắm rồi khi mà muôn thú, cây cỏ, con người còn trò chuyện được với nhau. Trên đồng cỏ rậm ven khu làng có một loài cây gọi là cây đa.
Đó là một thứ cây to, khỏe, lá của nó rậm rạp đến nỗi không một tia nắng nào có thể lọt qua được. Vào những ngày trời nắng nóng người ta thường nghỉ chân một lát và trò chuyện hàn huyên cùng cây dưới bóng cây mát rượi. Mọi người ai cũng biết rằng cây đa rất thông thái vì cây đã có tuổi, đã từng trải.
Một hôm, có một cậu bé chăn cừu ngồi nghỉ mát dưới gốc cây sau một ngày dài phơi mình dưới nắng cậu bé thấy người mệt mỏi và nóng bức. Một làn gió mơn man thổi thoa nhẹ lên tấm thân mỏi mệt của chú bé. Cậu bé bắt đầu thấy buồn ngủ. Vừa đặt mình xuống cậu bé bỗng ngước mắt nhìn lên những cành cây. Bấy giờ cậu bé bỗng thấy mình thật kiêu hãnh, cậu vẫn thường hay khoe với mọi người rằng cậu có tài chăn cừu và đàn cừu của cậu nhờ vậy mà lớn rất nhanh. Khi cậu bé phát hiện ra cây đa chỉ có những chùm quả rất nhỏ nó bắt đầu thấy ngạc nhiên. Cậu bắt đầu chế giễu: hư, một cái cây to khỏe thế này mà làm sao chỉ có những bông hoa những chùm quả bé tí tẹo thế kia, mọi người vẫn bảo là cái cây này thông thái lắm kia mà nhưng làm sao nó có thể thông thái khi mà quả của nó chỉ toàn bé xíu như vậy. Dĩ nhiên là cây đa nghe hết những lời của cậu bé nhưng cây vẫn im lặng và cành lá chỉ khẽ rung rinh đủ để cho gió cất lên khúc hát ru êm dịu. Cậu bé bắt đầu ngủ, cậu ngáy o o.... Cốc.
Quả đa nhỏ rụng chính giữa trán của cậu bé, nó bừng tỉnh nhưng càu nhàu: "gừm... người ta vừa mới chợp mắt được có một tí", rồi nó nhặt quả đa lên chưa hết chưa biết định làm gì với quả đa này bỗng nhiên cậu bé nghe thấy có tiếng cười khúc khích, cậu nghe thấy cây hỏi:
"Có đau không ?".
"Không nhưng mà làm người ta mất cả giấc ngủ".
"Đó là bài học cho cậu bé to đầu đấy. Cậu chẳng vừa nhạo tôi là chỉ sinh ra toàn những quả nhỏ xíu là gì".
"Tôi nhạo đấy tại sao người đời lại bảo bác là thông thái được nhỉ? Phá giấc ngủ trưa của người khác! Thế cũng là thông minh chắc! ".
Cây cười và nói: " này này anh bạn anh hãy nghe đây những chiếc lá của tôi cho cậu bóng mát để cậu lấy chỗ nghỉ ngơi. ừ thì cứ cho là quả của tôi nó bé đi chăng nữa nhưng chẳng lẽ cậu không thấy rằng tạo hóa hoạt động rất hoàn chỉnh đó sao. Cậu thử tưởng tượng xem, nếu quả của tôi to như quả dừa thì điều gì sẽ xảy ra khi nó rơi vào đầu cậu".
Cậu bé im thin thít: "ừ nhở". Câu chưa hề nghĩ đến điều này bao giờ cả.
Cây lại nhẹ nhàng tiếp lời: "Những người khiêm tốn có thể học hỏi rất nhiều điều từ việc quan sát những vật xung quanh đấy cậu bé ạ".
"Vâng bác đa bác cứ nói tiếp đi".
"Cậu hãy bắt đầu làm bạn với những gì ở quanh cậu. Chúng ta tất cả đều cần tới nhau. Cậu cứ nhìn bầy ong kia mà xem. Nhờ có ong mà hoa của tôi mới có thể trở thành quả. Thế còn bầy chim kia thì sao. Chúng làm tổ ngay giữa tán lá của tôi đây này. Những con chim bố mẹ kia phải làm việc vất vả cả ngày để bắt sâu nuôi con và cậu có biết việc làm đó có ý nghĩa gì với tôi không?".
"Không, có ý nghĩa gì vậy hả bác?".
"Sâu ăn lá chính vậy loài chim kia chính là những người bạn của tôi. Chúng còn giúp cả cậu nữa đấy, sở dĩ cừu của cậu có đủ lá và cỏ để ăn là vì chim chóc đã tiêu diệt hết các loài côn trùng và sâu bọ. Và chưa hết đâu cậu bé ạ!".
" Còn gì thế nữa hả bác đa".
"Cậu hãy nhìn xuống chân mình mà xem, những chiếc lá rụng tạo thành lớp thảm mục, những con sâu đào đất ngoi lên để ăn lá, chúng đào đất thành những lỗ nhỏ, nhờ đó không khí có thể vào được trong đất. Có không khí trong đất nên bộ rễ của tôi mới khỏe thế nào đấy. Rễ khỏe nên tôi cũng khỏe hơn. Nào thế bây giờ cậu trẻ đã hiểu chưa?".
"Cháu hiểu rồi thưa bác. Bác tha lỗi cho cháu nhé vì đã cười nhạo bác bác đa ạ".
" Không sao bây giờ cháu hãy ra dắt cừu về đi"
Có thể cậu bé chăn cừu không phải ngay sau đó sẽ trở nên khiêm tốn, học hỏi luôn được nhưng rõ ràng là cậu đã nhận ra người ta không thể sống lẻ loi phải không các bé."""

grade = 5

print(text)
for mode in ["long"]:

    summary, total_words, sum_words = summarize_text(
        text,
        grade,
        length_option=mode,
    )

    print("\n=======================")
    print("Mode:", mode)
    print("Số từ văn bản:", total_words)
    print("Số từ summary target:", sum_words)
    print("Summary:", summary)

Ngày xửa ngay xưa, xưa lắm rồi khi mà muôn thú, cây cỏ, con người còn trò chuyện được với nhau. Trên đồng cỏ rậm ven khu làng có một loài cây gọi là cây đa.
Đó là một thứ cây to, khỏe, lá của nó rậm rạp đến nỗi không một tia nắng nào có thể lọt qua được. Vào những ngày trời nắng nóng người ta thường nghỉ chân một lát và trò chuyện hàn huyên cùng cây dưới bóng cây mát rượi. Mọi người ai cũng biết rằng cây đa rất thông thái vì cây đã có tuổi, đã từng trải.
Một hôm, có một cậu bé chăn cừu ngồi nghỉ mát dưới gốc cây sau một ngày dài phơi mình dưới nắng cậu bé thấy người mệt mỏi và nóng bức. Một làn gió mơn man thổi thoa nhẹ lên tấm thân mỏi mệt của chú bé. Cậu bé bắt đầu thấy buồn ngủ. Vừa đặt mình xuống cậu bé bỗng ngước mắt nhìn lên những cành cây. Bấy giờ cậu bé bỗng thấy mình thật kiêu hãnh, cậu vẫn thường hay khoe với mọi người rằng cậu có tài chăn cừu và đàn cừu của cậu nhờ vậy mà lớn rất nhanh. Khi cậu bé phát hiện ra cây đa chỉ có những chùm quả rất nhỏ nó bắt đầu thấy ngạc nhiên. 

In [7]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from underthesea import word_tokenize
import torch

# ---------------------------
# Tiền xử lý tiếng Việt
# ---------------------------
def preprocess_vi(text):
    text = text.strip()
    text = text.replace("\n", " ")
    tokens = word_tokenize(text, format="text")
    return tokens


# ---------------------------
# Tính ROUGE
# ---------------------------
def compute_rouge(candidate, reference):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=False
    )

    scores = scorer.score(reference, candidate)

    rouge_results = {
        "rouge1_precision": scores['rouge1'].precision,
        "rouge1_recall": scores['rouge1'].recall,
        "rouge1_f1": scores['rouge1'].fmeasure,
        "rougeL_precision": scores['rougeL'].precision,
        "rougeL_recall": scores['rougeL'].recall,
        "rougeL_f1": scores['rougeL'].fmeasure,
    }

    return rouge_results


# ---------------------------
# Tính BERTScore (PhoBERT)
# ---------------------------
from bert_score import score
import torch

def compute_bertscore(candidate, reference):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = score(
        [candidate],
        [reference],
        lang="vi",          # QUAN TRỌNG
        model_type=None,    # KHÔNG ghi vinai/phobert-base
        device=device
    )

    return {
        "bertscore_precision": P.item(),
        "bertscore_recall": R.item(),
        "bertscore_f1": F1.item()
    }



# ---------------------------
# Hàm tổng hợp
# ---------------------------
def evaluate_summary(candidate, reference):
    candidate_proc = preprocess_vi(candidate)
    reference_proc = preprocess_vi(reference)

    candidate_text = " ".join(candidate_proc)
    reference_text = " ".join(reference_proc)

    results = {}
    results.update(compute_rouge(candidate_text, reference_text))
    results.update(compute_bertscore(candidate_text, reference_text))

    return results


import pandas as pd
import numpy as np

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\data_test_main_summodel2.xlsx")
# Danh sách cột thang đo
metric_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "bertscore_precision", "bertscore_recall", "bertscore_f1"
]

# Nếu chưa có cột thì tạo
for col in metric_cols:
    if col not in df.columns:
        df[col] = np.nan

# =========================
# TÍNH ĐIỂM TỪNG ROW
# =========================
for idx, row in df.iterrows():
    content = row.get("content", "")
    summary = row.get("summary_model", "")

    # Bỏ qua nếu thiếu dữ liệu
    if not isinstance(content, str) or not isinstance(summary, str):
        continue
    if content.strip() == "" or summary.strip() == "":
        continue

    try:
        scores = evaluate_summary(summary, content)

        for k, v in scores.items():
            df.at[idx, k] = float(v)

    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue
    
df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\data_test_main_summodel2_score2.xlsx", index=False)


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:
# ============================================
# LOAD MODEL ĐÃ TRAIN
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import pandas as pd 

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\diengiaitest.xlsx")

MODEL_PATH = "../../Model_DG/vit5_grade_summary"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# ============================================
# HÀM TÓM TẮT 1 VĂN BẢN
# ============================================

def summarize_text(content, grade,
                   max_input_len=512,
                   max_target_len=256):

    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {content}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_target_len,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary


# ============================================
# TEST VỚI 1 VĂN BẢN BẤT KỲ
# ============================================

grade = 5

df["sum"] = ""  

for i in range(len(df)):
    text = df.iloc[i]["content"]
    summary = summarize_text(text, grade)
    df.iloc[i, df.columns.get_loc("sum")] = summary

df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\diengiaitest_sum.xlsx")


In [4]:
import pandas as pd 

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\diengiaitest_score.xlsx")
df.describe()

,Unnamed: 0,summary,grade,rouge1_precision,rouge1_recall,rouge1_f1,rougeL_precision,rougeL_recall,rougeL_f1,bertscore_precision,bertscore_recall,bertscore_f1
count,5.000000,0.0,5.000000,5.0,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,2.000000,NaN,3.000000,1.0,0.552875,0.704454,0.906374,0.502241,0.639062,0.874836,0.867130,0.870922
std,1.581139,NaN,1.581139,0.0,0.135036,0.109772,0.065762,0.139124,0.118406,0.021327,0.033938,0.027654
min,0.000000,NaN,1.000000,1.0,0.393293,0.564551,0.828571,0.381098,0.547046,0.837162,0.806988,0.821799
25%,1.000000,NaN,2.000000,1.0,0.480149,0.648785,0.844961,0.405707,0.548198,0.878655,0.877054,0.878797
50%,2.000000,NaN,3.000000,1.0,0.561224,0.718954,0.927083,0.465015,0.595705,0.884638,0.878940,0.882077
75%,3.000000,NaN,4.000000,1.0,0.572565,0.728192,0.962264,0.530815,0.675095,0.886568,0.883638,0.884138
max,4.000000,NaN,5.000000,1.0,0.757143,0.861789,0.968992,0.728571,0.829268,0.887158,0.889030,0.887797


In [16]:
import pandas as pd 

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\Score model\Book1_score.xlsx")
df.describe()

,grade,rouge1_precision,rouge1_recall,rouge1_f1,rougeL_precision,rougeL_recall,rougeL_f1,bertscore_precision,bertscore_recall,bertscore_f1
count,2000.0,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,5.0,0.996919,0.347841,0.482581,0.905997,0.308463,0.429889,0.873448,0.865801,0.869564
std,0.0,0.012236,0.201106,0.218888,0.069523,0.171599,0.186263,0.027171,0.034485,0.030563
min,5.0,0.784173,0.028853,0.056087,0.565972,0.028853,0.056087,0.740772,0.691704,0.722325
25%,5.0,1.000000,0.174485,0.297126,0.868203,0.162896,0.277680,0.870202,0.864113,0.867360
50%,5.0,1.000000,0.328950,0.495052,0.920839,0.290840,0.435697,0.879970,0.875781,0.877694
75%,5.0,1.000000,0.492679,0.659733,0.958265,0.433918,0.577768,0.888052,0.884213,0.885779
max,5.0,1.000000,0.989011,0.953642,1.000000,0.900000,0.918367,0.934468,0.924712,0.929564


In [1]:
"""
Batch Abstractive Summarization — All-in-one Script
=====================================================
Input  : E:\\Project_NguyenMinhVu_2211110063\\Source\\datasets\\dataset abstractive\\Data_test_1000.xlsx
Output : E:\\Project_NguyenMinhVu_2211110063\\Source\\datasets\\dataset abstractive\\Data_test_1000_sum.xlsx

Chạy nhanh:
    python batch_summarize.py
    python batch_summarize.py --mode beam --length short
    python batch_summarize.py --resume                        # tiếp tục từ lần dừng trước
    python batch_summarize.py --content-col "noi_dung" --grade-col "lop"
"""

# ===========================================================
# IMPORTS
# ===========================================================
import gc
import re
import unicodedata
import argparse
import sys
import time
from pathlib import Path
from typing import Optional, Literal

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx

try:
    from vncorenlp import VnCoreNLP
    VNCORENLP_AVAILABLE = True
except ImportError:
    VNCORENLP_AVAILABLE = False

# ===========================================================
# ▶▶ CẤU HÌNH — chỉnh tại đây nếu không dùng CLI ◀◀
# ===========================================================
INPUT_PATH    = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\datatest\datatest_filtered_sum2_model.xlsx"
OUTPUT_PATH   = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\datatest\datatest_filtered_sum2_model2.xlsx"
MODEL_PATH    = r"E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_DG_ver2\vit5_grade_summary"
VNCORENLP_JAR = r"E:\Project_NguyenMinhVu_2211110063\Source\ai\VnCoreNLP-master\VnCoreNLP-1.1.1.jar"

COL_CONTENT   = "content"      # cột văn bản gốc
COL_GRADE     = "grade"        # cột lớp học (để "" nếu file không có)
COL_OUT       = "summary_model"  # cột kết quả — ghi vào cột có sẵn trong file

DEFAULT_GRADE    = 5        # lớp mặc định khi không có cột grade
DEFAULT_MODE     = "sample"   # "sample" | "beam"
DEFAULT_LENGTH   = "long"   # "short"  | "medium" | "long"
CHECKPOINT_EVERY = 50       # lưu checkpoint sau mỗi N dòng

# ===========================================================
# CONSTANTS
# ===========================================================
WORD_LIMIT               = 768
MAX_WORDS                = 768
SMALL_MODEL_CHAR_THRESH  = 100
SMALL_MODEL_GRADE        = 1

# ===========================================================
# ABSTRACTER AGENT
# ===========================================================

class AbstracterAgent:
    """Fine-tuned ViT5 grade-based abstractive summarizer."""

    def __init__(
        self,
        model_path: str = MODEL_PATH,
        small_model_path: Optional[str] = None,
        max_input_len: int = 768,
        max_target_len: int = 256,
        num_beams: int = 4,
        no_repeat_ngram_size: int = 3,
        annotator=None,
        vncorenlp=None,
        model_simcse: Optional[SentenceTransformer] = None,
    ):
        self.model_path           = model_path
        self.small_model_path     = small_model_path
        self._small_tokenizer     = None
        self._small_model         = None
        self.max_input_len        = max_input_len
        self.max_target_len       = max_target_len
        self.num_beams            = num_beams
        self.no_repeat_ngram_size = no_repeat_ngram_size
        self.annotator            = annotator
        self.vncorenlp            = vncorenlp
        self.device               = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_simcse         = model_simcse or SentenceTransformer(
            "VoVanPhuc/sup-SimCSE-VietNamese-phobert-base"
        )
        self._load_model()

    # ── model loading ────────────────────────────────────────

    def _load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path, use_fast=False)
        self.model     = AutoModelForSeq2SeqLM.from_pretrained(self.model_path)
        self.model.to(self.device)
        self.model.eval()

    def _load_small_model(self):
        if not self.small_model_path or self._small_model is not None:
            return
        self._small_tokenizer = AutoTokenizer.from_pretrained(self.small_model_path, use_fast=False)
        self._small_model     = AutoModelForSeq2SeqLM.from_pretrained(self.small_model_path)
        self._small_model.to(self.device)
        self._small_model.eval()

    def _unload_small_model(self):
        if self._small_model is None:
            return
        del self._small_model
        self._small_model, self._small_tokenizer = None, None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _seq2seq_for_request(self, content: str, grade: int):
        if (
            self.small_model_path
            and grade == SMALL_MODEL_GRADE
            and len(content.strip()) < SMALL_MODEL_CHAR_THRESH
        ):
            self._load_small_model()
            return self._small_tokenizer, self._small_model, True
        return self.tokenizer, self.model, False

    # ── text utilities ───────────────────────────────────────

    def normalize_text(self, text: str) -> str:
        text = text.replace("_", " ")
        text = re.sub(r'\s+([,\.!?;:\)\]])', r'\1', text)
        text = re.sub(r'([,\.!?;:])(?!\s)(?!\d)', r'\1 ', text)
        text = re.sub(r'([\(\[])\s+', r'\1', text)
        return re.sub(r' +', ' ', text).strip()

    def sentence_split(self, text: str) -> list:
        if self.vncorenlp is None:
            sents = re.split(r'(?<=[.!?])\s+', text.strip())
            return [self.normalize_text(s) for s in sents if s.strip()]
        try:
            sents = []
            for sent in self.vncorenlp.annotate(text)["sentences"]:
                raw = " ".join(w["form"] for w in sent)
                sents.append(self.normalize_text(raw))
            return sents
        except Exception:
            sents = re.split(r'(?<=[.!?])\s+', text.strip())
            return [self.normalize_text(s) for s in sents if s.strip()]

    def count_words(self, text: str) -> int:
        return len(text.split())

    # ── extractive pre-processing (TextRank + LexRank + MMR) ─

    def _tfidf_sim(self, sentences):
        tfidf = TfidfVectorizer().fit_transform(sentences)
        return cosine_similarity(tfidf)

    def textrank(self, sentences):
        sim   = self._tfidf_sim(sentences)
        graph = nx.from_numpy_array(sim)
        return nx.pagerank(graph)

    def lexrank(self, sentences):
        sim    = self._tfidf_sim(sentences)
        scores = sim.sum(axis=1)
        return dict(enumerate(scores))

    def filter_by_ratio(self, sentences, ratio=0.7):
        tr       = self.textrank(sentences)
        lr       = self.lexrank(sentences)
        combined = {i: 0.5 * tr[i] + 0.5 * lr[i] for i in range(len(sentences))}
        k        = max(1, int(len(sentences) * ratio))
        top_ids  = sorted(combined, key=combined.get, reverse=True)[:k]
        return [sentences[i] for i in sorted(top_ids)]

    def phobert_scoring(self, sentences, full_text):
        sent_emb = self.model_simcse.encode(sentences, normalize_embeddings=True)
        doc_emb  = self.model_simcse.encode([full_text], normalize_embeddings=True)[0]
        return sent_emb @ doc_emb

    def mmr(self, sentences, scores, lambda_=0.8, top_k=5):
        sent_emb   = self.model_simcse.encode(sentences, normalize_embeddings=True)
        selected, candidates = [], list(range(len(sentences)))
        for _ in range(min(top_k, len(candidates))):
            mmr_scores = [
                (i, lambda_ * scores[i] - (1 - lambda_) * max(
                    [sent_emb[i] @ sent_emb[j] for j in selected], default=0
                ))
                for i in candidates
            ]
            best = max(mmr_scores, key=lambda x: x[1])[0]
            selected.append(best)
            candidates.remove(best)
        return [sentences[i] for i in sorted(selected)]

    def extract_summary(self, text: str, max_words: int,
                        filter_ratio=0.7, mmr_ratio=0.5, lambda_=0.8) -> str:
        sentences = self.sentence_split(text)
        if len(sentences) < 3:
            result = self.normalize_text(text)
            if max_words and self.count_words(result) > max_words:
                result = " ".join(result.split()[:max_words])
            return result
        filtered = self.filter_by_ratio(sentences, ratio=filter_ratio)
        scores   = self.phobert_scoring(filtered, text)
        top_k    = max(1, int(len(filtered) * mmr_ratio))
        while top_k >= 1:
            summary = " ".join(self.mmr(filtered, scores, lambda_=lambda_, top_k=top_k))
            if not max_words or self.count_words(summary) <= max_words:
                break
            top_k -= 1
        else:
            summary = " ".join(summary.split()[:max_words])
        return summary

    def smart_summary(self, text: str) -> str:
        """Cắt bớt đầu vào nếu quá dài để vừa context window."""
        if self.count_words(text) > WORD_LIMIT:
            return self.extract_summary(text, max_words=MAX_WORDS)
        return text

    # ── length helpers ───────────────────────────────────────

    def _count_words(self, text: str) -> int:
        return len(text.split()) if isinstance(text, str) else 0

    def _word_to_token_estimate(self, n: int) -> int:
        return int(n * 1.3) if n > 0 else 0

    def _clean_summary(self, summary: str) -> str:
        if not isinstance(summary, str):
            return ""
        last = summary.rfind(".")
        return summary[:last + 1].strip() if last != -1 else summary.strip()

    # ── post-processing ──────────────────────────────────────

    def vietnamese_text_normalization(self, text: str) -> str:
        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s+([,.!?;:])", r"\1", text)
        text = re.sub(r"([,.!?;:])([^\s])", r"\1 \2", text)
        text = re.sub(
            r"([a-zàáạảãăắằẳẵặâấầẩẫậđèéẹẻẽêếềểễệìíịỉĩòóọỏõôốồổỗộơớờởỡợùúụủũưứừửữựỳýỵỷỹ])"
            r"([A-ZÀÁẠẢÃĂẮẰẲẴẶÂẤẦẨẪẬĐ])", r"\1 \2", text
        )
        words = [w.capitalize() if w.isupper() and len(w) > 2 else w for w in text.split()]
        sents = re.split(r'(?<=[.!?]) +', " ".join(words))
        return " ".join(s[0].upper() + s[1:] if s else s for s in sents).strip()

    def extract_entities(self, text: str) -> list:
        if self.annotator is None or not isinstance(text, str) or not text.strip():
            return []
        entities = set()
        try:
            for sent in self.annotator.annotate(text).get("sentences", []):
                cur = []
                for token in sent:
                    label = token.get("nerLabel", "O")
                    if label.startswith("B-"):
                        if cur:
                            entities.add(" ".join(cur))
                        cur = [token["form"]]
                    elif label.startswith("I-") and cur:
                        cur.append(token["form"])
                    else:
                        if cur:
                            entities.add(" ".join(cur))
                        cur = []
                if cur:
                    entities.add(" ".join(cur))
        except Exception:
            pass
        return list(entities)

    def correct_entities(self, summary: str, entities: list, content: str) -> str:
        if not isinstance(summary, str) or not summary.strip():
            return summary
        for entity in sorted(entities, key=len, reverse=True):
            raw = entity.strip().replace("_", " ").strip()
            if not raw:
                continue
            m = re.search(re.escape(raw), content, flags=re.IGNORECASE) if content else None
            canonical   = m.group(0) if m else " ".join(w.capitalize() for w in raw.split())
            tokens      = re.split(r"\s+", raw)
            pattern_str = (r"[_ ]*".join(re.escape(t) for t in tokens)
                           if len(tokens) > 1 else re.escape(tokens[0]))
            summary = re.compile(pattern_str, re.IGNORECASE).sub(canonical, summary)
        return re.sub(r"\s+", " ", summary.replace("_", " ")).strip()

    # ── public API ───────────────────────────────────────────

    def summarize(
        self,
        content: str,
        grade: int,
        max_input_len: Optional[int] = None,
        max_target_len: Optional[int] = None,
        mode: str = "sample",
        length_option: Literal["short", "medium", "long"] = "medium",
    ) -> str:
        if not content or not content.strip():
            return "Nội dung trống."

        max_input_len  = max_input_len  or self.max_input_len
        max_target_len = max_target_len or self.max_target_len

        total_words    = self._count_words(content)
        ratio          = {"short": 0.30, "long": 0.60}.get(length_option, 0.40)
        summary_words  = int(total_words * ratio)

        target_max = min(self._word_to_token_estimate(summary_words), max_target_len)
        target_max = max(target_max, 1)
        target_min = max(1, min(int(target_max * 0.3), target_max))

        tokenizer, seq2seq_model, used_small = self._seq2seq_for_request(content, grade)
        try:
            prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {content}"
            inputs = tokenizer(
                prompt, return_tensors="pt", truncation=True, max_length=max_input_len
            ).to(self.device)

            with torch.no_grad():
                if mode == "beam":
                    output_ids = seq2seq_model.generate(
                        **inputs,
                        min_new_tokens=target_min,
                        max_new_tokens=target_max,
                        num_beams=self.num_beams,
                        length_penalty=1.2,
                        no_repeat_ngram_size=self.no_repeat_ngram_size,
                        early_stopping=True,
                    )
                else:
                    output_ids = seq2seq_model.generate(
                        **inputs,
                        min_new_tokens=target_min,
                        max_new_tokens=target_max,
                        do_sample=True,
                        top_k=50,
                        top_p=0.9,
                        temperature=0.8,
                        no_repeat_ngram_size=self.no_repeat_ngram_size,
                    )

            summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            del output_ids, inputs

            summary  = self._clean_summary(summary)
            summary  = self.vietnamese_text_normalization(summary)
            entities = self.extract_entities(content)
            if entities:
                summary = self.correct_entities(summary, entities, content)
            return summary.strip()
        finally:
            if used_small:
                self._unload_small_model()

    def run(
        self,
        content: str,
        grade: int = 5,
        mode: str = "sample",
        length_option: Literal["short", "medium", "long"] = "medium",
    ) -> str:
        try:
            return self.summarize(self.smart_summary(content), grade,
                                  mode=mode, length_option=length_option)
        except Exception as e:
            return f"[Error] {e}"


# ===========================================================
# BATCH RUNNER
# ===========================================================

def parse_args():
    p = argparse.ArgumentParser(description="Batch abstractive summarization từ Excel")
    p.add_argument("--input",          default=INPUT_PATH)
    p.add_argument("--output",         default=OUTPUT_PATH)
    p.add_argument("--model",          default=MODEL_PATH)
    p.add_argument("--vncorenlp",      default=VNCORENLP_JAR)
    p.add_argument("--content-col",    default=COL_CONTENT)
    p.add_argument("--grade-col",      default=COL_GRADE)
    p.add_argument("--out-col",        default=COL_OUT)
    p.add_argument("--default-grade",  type=int, default=DEFAULT_GRADE)
    p.add_argument("--mode",           default=DEFAULT_MODE, choices=["sample", "beam"])
    p.add_argument("--length",         default=DEFAULT_LENGTH, choices=["short", "medium", "long"])
    p.add_argument("--checkpoint",     type=int, default=CHECKPOINT_EVERY)
    p.add_argument("--resume",         action="store_true",
                   help="Tiếp tục: bỏ qua dòng đã có kết quả trong file output")
    # parse_known_args bỏ qua các tham số lạ do Jupyter tự thêm (vd: --f=kernel-xxx.json)
    args, _ = p.parse_known_args()
    return args


def load_vncorenlp(jar_path: str):
    if not VNCORENLP_AVAILABLE:
        print("[INFO] Thư viện vncorenlp chưa cài — dùng fallback regex.")
        return None, None
    if not jar_path or not Path(jar_path).exists():
        print("[INFO] Không tìm thấy VnCoreNLP jar — dùng fallback regex.")
        return None, None
    try:
        obj = VnCoreNLP(jar_path, annotators="wseg,pos,ner", max_heap_size="-Xmx2g")
        print("[INFO] VnCoreNLP đã tải.")
        return obj, obj
    except Exception as e:
        print(f"[WARN] Lỗi VnCoreNLP: {e} — dùng fallback regex.")
        return None, None


def load_dataframe(args) -> pd.DataFrame:
    out_path = Path(args.output)
    if args.resume and out_path.exists():
        df = pd.read_excel(args.output, dtype=str)
        print(f"[INFO] Resume từ output: {args.output} ({len(df)} dòng)")
    else:
        df = pd.read_excel(args.input, dtype=str)
        print(f"[INFO] Đọc input: {args.input} ({len(df)} dòng)")
    if args.out_col not in df.columns:
        df[args.out_col] = ""
    return df


def run_batch(args):
    bar = "=" * 62
    print(bar)
    print("  BATCH ABSTRACTIVE SUMMARIZATION")
    print(bar)
    print(f"  Input        : {args.input}")
    print(f"  Output       : {args.output}")
    print(f"  Mode/Length  : {args.mode} / {args.length}")
    print(f"  Content col  : {args.content_col}")
    print(f"  Grade col    : {args.grade_col or '(không có)'}")
    print(f"  Output col   : {args.out_col}")
    print(f"  Default grade: {args.default_grade}")
    print(f"  Resume       : {args.resume}")
    print(bar)

    annotator, vncorenlp = load_vncorenlp(args.vncorenlp)

    print("[INFO] Đang tải model ViT5 và SimCSE...")
    agent = AbstracterAgent(
        model_path  = args.model,
        annotator   = annotator,
        vncorenlp   = vncorenlp,
    )
    print(f"[INFO] Thiết bị: {agent.device}\n")

    df = load_dataframe(args)

    if args.content_col not in df.columns:
        sys.exit(
            f"[ERROR] Cột '{args.content_col}' không tồn tại.\n"
            f"  Các cột hiện có: {list(df.columns)}"
        )

    total     = len(df)
    processed = skipped = errors = 0
    t0        = time.time()

    for idx in range(total):
        row = df.iloc[idx]

        # Resume: bỏ qua dòng đã có kết quả
        if args.resume:
            val = str(row.get(args.out_col, "")).strip()
            if val and val.lower() not in ("", "nan"):
                skipped += 1
                continue

        content = str(row.get(args.content_col, "")).strip()
        if not content or content.lower() == "nan":
            df.at[idx, args.out_col] = ""
            continue

        # Lấy grade
        grade = args.default_grade
        if args.grade_col and args.grade_col in df.columns:
            try:
                grade = int(float(str(row[args.grade_col])))
            except (ValueError, TypeError):
                grade = args.default_grade

        # Tóm tắt
        try:
            result = agent.run(content, grade=grade,
                               mode=args.mode, length_option=args.length)
        except Exception as e:
            result = f"[ERROR] {e}"
            errors += 1

        df.at[idx, args.out_col] = result
        processed += 1

        # Progress log
        elapsed = time.time() - t0
        avg_sec = elapsed / processed
        eta_min = avg_sec * (total - skipped - processed) / 60
        print(
            f"  [{skipped + processed:>5}/{total}] row={idx} | lớp {grade} | "
            f"{len(content)}→{len(result)} ký tự | ETA {eta_min:.1f} phút",
            flush=True,
        )

        # Checkpoint
        if processed % args.checkpoint == 0:
            df.to_excel(args.output, index=False)
            print(f"  ✔ Checkpoint → {args.output}")

    # Lưu cuối
    df.to_excel(args.output, index=False)

    total_min = (time.time() - t0) / 60
    print(f"\n{bar}")
    print("  HOÀN TẤT")
    print(f"  Đã xử lý  : {processed} dòng")
    print(f"  Bỏ qua    : {skipped} dòng")
    print(f"  Lỗi       : {errors} dòng")
    print(f"  Thời gian : {total_min:.1f} phút")
    print(f"  Lưu tại   : {args.output}")
    print(bar)

    if annotator:
        try:
            annotator.close()
        except Exception:
            pass


# ===========================================================
# ENTRY POINT 
# ===========================================================
if __name__ == "__main__":
    run_batch(parse_args())

c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\networkx\utils\backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


  BATCH ABSTRACTIVE SUMMARIZATION
  Input        : E:\Project_NguyenMinhVu_2211110063\Source\datasets\datatest\datatest_filtered_sum2_model.xlsx
  Output       : E:\Project_NguyenMinhVu_2211110063\Source\datasets\datatest\datatest_filtered_sum2_model2.xlsx
  Mode/Length  : sample / long
  Content col  : content
  Grade col    : grade
  Output col   : summary_model
  Default grade: 5
  Resume       : False
[INFO] VnCoreNLP đã tải.
[INFO] Đang tải model ViT5 và SimCSE...


No sentence-transformers model found with name VoVanPhuc/sup-SimCSE-VietNamese-phobert-base. Creating a new one with mean pooling.


[INFO] Thiết bị: cuda

[INFO] Đọc input: E:\Project_NguyenMinhVu_2211110063\Source\datasets\datatest\datatest_filtered_sum2_model.xlsx (4982 dòng)
  [    1/4982] row=0 | lớp 2 | 398→186 ký tự | ETA 208.4 phút
  [    2/4982] row=1 | lớp 2 | 479→239 ký tự | ETA 185.2 phút
  [    3/4982] row=2 | lớp 2 | 300→144 ký tự | ETA 159.8 phút
  [    4/4982] row=3 | lớp 2 | 438→273 ký tự | ETA 156.5 phút
  [    5/4982] row=4 | lớp 2 | 285→138 ký tự | ETA 147.7 phút
  [    6/4982] row=5 | lớp 2 | 283→150 ký tự | ETA 138.5 phút
  [    7/4982] row=6 | lớp 2 | 398→198 ký tự | ETA 139.2 phút
  [    8/4982] row=7 | lớp 2 | 473→228 ký tự | ETA 142.8 phút
  [    9/4982] row=8 | lớp 2 | 369→200 ký tự | ETA 140.9 phút
  [   10/4982] row=9 | lớp 2 | 267→140 ký tự | ETA 137.1 phút
  [   11/4982] row=10 | lớp 2 | 307→210 ký tự | ETA 133.4 phút
  [   12/4982] row=11 | lớp 2 | 339→200 ký tự | ETA 131.5 phút
  [   13/4982] row=12 | lớp 2 | 475→230 ký tự | ETA 133.4 phút
  [   14/4982] row=13 | lớp 2 | 251→153 ký t

In [ ]:
from rouge_score import rouge_scorer
from bert_score import BERTScorer
from underthesea import word_tokenize
from transformers import AutoTokenizer
import torch
import pandas as pd
import numpy as np

# ---------------------------
# Tiền xử lý — chỉ dùng cho ROUGE
# ---------------------------
def preprocess_vi(text: str) -> str:
    return word_tokenize(text.strip().replace("\n", " "), format="text")


# ---------------------------
# Tính ROUGE
# ---------------------------
def compute_rouge(candidate: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=False
    )
    scores = scorer.score(reference, candidate)
    return {
        "rouge1_f1"        : scores['rouge1'].fmeasure,
        "rouge2_f1"        : scores['rouge2'].fmeasure,
        "rougeL_f1"        : scores['rougeL'].fmeasure,
    }


# ---------------------------
# Truncate an toàn — PhoBERT: max_position_embeddings = 258
# model_max_length của tokenizer rất lớn → BERTScore không tự cắt đủ.
# decode(encode(...)) có thể dài hơn encode(...) → phải kiểm tra round-trip.
# ---------------------------
def safe_truncate(text: str, tokenizer: AutoTokenizer,
                  max_tokens: int = 250) -> str:
    text = (text or "").strip()
    if not text:
        return ""
    n = max_tokens
    while n >= 16:
        ids = tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=True,
            max_length=n,
        )
        s = tokenizer.decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        chk = tokenizer.encode(
            s,
            add_special_tokens=True,
            truncation=True,
            max_length=max_tokens + 16,
        )
        if len(chk) <= max_tokens:
            return s
        n -= 1
    ids = tokenizer.encode(
        text, add_special_tokens=True, truncation=True, max_length=16
    )
    return tokenizer.decode(
        ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )


# ---------------------------
# Tính BERTScore
# ---------------------------
def compute_bertscore(candidate: str, reference: str,
                      scorer: BERTScorer,
                      tokenizer: AutoTokenizer,
                      prefix: str = "") -> dict:
    cand_trunc = safe_truncate(candidate, tokenizer)
    ref_trunc  = safe_truncate(reference, tokenizer)

    # Đảm bảo không rỗng sau truncate
    if not cand_trunc.strip():
        cand_trunc = candidate[:100]
    if not ref_trunc.strip():
        ref_trunc  = reference[:100]

    P, R, F1 = scorer.score([cand_trunc], [ref_trunc])

    key = f"bertscore_{prefix}" if prefix else "bertscore"
    return {
        f"{key}_f1"        : F1.item(),
    }


# ---------------------------
# Hàm tổng hợp
# ROUGE: tóm tắt tham chiếu vs tóm tắt model
# BERTScore: tóm tắt model vs văn bản gốc (content)
# ---------------------------
def evaluate_summary(candidate: str, reference: str,
                     scorer: BERTScorer,
                     tokenizer: AutoTokenizer) -> dict:
    results = compute_rouge(preprocess_vi(candidate), preprocess_vi(reference))
    results.update(
        compute_bertscore(candidate, reference, scorer, tokenizer, prefix="")
    )
    return results


# =========================
# LOAD MODEL & TOKENIZER MỘT LẦN
# =========================
PHOBERT_NAME = "vinai/phobert-base"

print("[INFO] Đang tải PhoBERT tokenizer...")
phobert_tokenizer = AutoTokenizer.from_pretrained(PHOBERT_NAME, use_fast=False)

print("[INFO] Đang tải BERTScorer...")
bert_scorer = BERTScorer(
    model_type=PHOBERT_NAME,
    num_layers=12,
    lang="vi",
    device="cpu",
    batch_size=8,
)
print("[INFO] Tải xong.\n")


# =========================
# BATCH EVALUATION
# =========================
data = pd.read_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\data_test_main_summodel2.xlsx"
)

df = data[:50].copy()

REF_COL     = "summary_ref"
CAND_COL    = "summary_model"

metric_cols = ["rouge1_f1", "rouge2_f1", "rougeL_f1", "bertscore_f1"]
for col in metric_cols:
    if col not in df.columns:
        df[col] = np.nan

for idx, row in df.iterrows():
    candidate = row.get(CAND_COL,    "")
    reference = row.get(REF_COL,     "")

    if not all(isinstance(x, str) for x in [candidate, reference]):
        continue
    if any(x.strip() == "" for x in [candidate, reference]):
        continue

    try:
        scores = evaluate_summary(candidate, reference, bert_scorer, phobert_tokenizer)
        for k, v in scores.items():
            df.at[idx, k] = float(v)
        if idx % 10 == 0:
            print(f"[{idx}/{len(df)}] done")
    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue

df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\data_test_main_summodel2_score.xlsx", index=False)

[INFO] Đang tải PhoBERT tokenizer...


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[INFO] Đang tải BERTScorer...
[INFO] Tải xong.

[0/50] done
[10/50] done
[20/50] done
[30/50] done
[40/50] done

=== Điểm trung bình ===
rouge1_precision       0.7850
rouge1_recall          0.7075
rouge1_f1              0.7214
rouge2_precision       0.4113
rouge2_recall          0.3573
rouge2_f1              0.3703
rougeL_precision       0.4124
rougeL_recall          0.3652
rougeL_f1              0.3751
bertscore_precision    0.4832
bertscore_recall       0.4756
bertscore_f1           0.4792


In [ ]:
import pandas as pd
from bert_score import BERTScorer

# Đọc dữ liệu (ví dụ file CSV)
df = pd.read_excel("E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\data_test_main_summodel2.xlsx")

# Lấy danh sách câu
references = df["content"].astype(str).tolist()
candidates = df["summary_model"].astype(str).tolist()

# Khởi tạo scorer (tiếng Việt nên dùng multilingual model)
scorer = BERTScorer(
    model_type="bert-base-multilingual-cased",
    lang="vi",
    rescale_with_baseline=False
)

# Tính BERTScore
P, R, F1 = scorer.score(candidates, references)

# Gán kết quả vào DataFrame
df["bert_precision"] = P.tolist()
df["bert_recall"] = R.tolist()
df["bert_f1"] = F1.tolist()

# In kết quả trung bình
print("Average BERTScore:")
print(f"Precision: {df['bert_precision'].mean():.4f}")
print(f"Recall:    {df['bert_recall'].mean():.4f}")
print(f"F1:        {df['bert_f1'].mean():.4f}")

# Lưu lại nếu cần
df.to_excel("E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\data_test_main_summodel2_score4.xlsx", index=False)

c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Average BERTScore:
Precision: 0.7462
Recall:    0.6868
F1:        0.7152


In [1]:
import pandas as pd
from bert_score import score
import torch

# Đọc dữ liệu
df = pd.read_excel("E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_test_1000_summary.xlsx")


# Lấy hai cột cần tính
references = df["content"].tolist()       # văn bản gốc (reference)
candidates = df["summary_end"].tolist() # tóm tắt của model (candidate/hypothesis)

# Tính BERTScore
# lang="vi" nếu văn bản tiếng Việt, "en" nếu tiếng Anh
P, R, F1 = score(
    cands=candidates,
    refs=references,
    lang="vi",              # đổi thành "en" nếu là tiếng Anh
    model_type=None,        # để None thì tự chọn model mặc định theo lang
    verbose=True
)

# Chuyển về numpy để dễ xử lý
df["bertscore_precision"] = P.numpy()
df["bertscore_recall"]    = R.numpy()
df["bertscore_f1"]        = F1.numpy()

# In kết quả trung bình
print(f"BERTScore trung bình:")
print(f"  Precision : {P.mean().item():.4f}")
print(f"  Recall    : {R.mean().item():.4f}")
print(f"  F1        : {F1.mean().item():.4f}")

# Lưu kết quả
df.to_excel("E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_test_1000_summary_score.xlsx", index=False)

print("\nĐã lưu kết quả vào result_bertscore.csv")

calculating scores...
computing bert embedding.


  0%|          | 0/16 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 50.56 seconds, 19.78 sentences/sec
BERTScore trung bình:
  Precision : 0.8807
  Recall    : 0.8056
  F1        : 0.8406

Đã lưu kết quả vào result_bertscore.csv


In [ ]:
from rouge_score import rouge_scorer
from bert_score import BERTScorer
from underthesea import word_tokenize
from transformers import AutoTokenizer
import torch
import pandas as pd
import numpy as np

# ---------------------------
# Tiền xử lý — chỉ dùng cho ROUGE
# ---------------------------
def preprocess_vi(text: str) -> str:
    return word_tokenize(text.strip().replace("\n", " "), format="text")


# ---------------------------
# Tính ROUGE
# ---------------------------
def compute_rouge(candidate: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=False
    )
    scores = scorer.score(reference, candidate)
    return {
        "rouge1_precision" : scores['rouge1'].precision,
        "rouge1_recall"    : scores['rouge1'].recall,
        "rouge1_f1"        : scores['rouge1'].fmeasure,
        "rouge2_precision" : scores['rouge2'].precision,
        "rouge2_recall"    : scores['rouge2'].recall,
        "rouge2_f1"        : scores['rouge2'].fmeasure,
        "rougeL_precision" : scores['rougeL'].precision,
        "rougeL_recall"    : scores['rougeL'].recall,
        "rougeL_f1"        : scores['rougeL'].fmeasure,
    }


# ---------------------------
# Truncate an toàn — PhoBERT: max_position_embeddings = 258
# model_max_length của tokenizer rất lớn → BERTScore không tự cắt đủ.
# decode(encode(...)) có thể dài hơn encode(...) → phải kiểm tra round-trip.
# ---------------------------
def safe_truncate(text: str, tokenizer: AutoTokenizer,
                  max_tokens: int = 250) -> str:
    text = (text or "").strip()
    if not text:
        return ""
    n = max_tokens
    while n >= 16:
        ids = tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=True,
            max_length=n,
        )
        s = tokenizer.decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        chk = tokenizer.encode(
            s,
            add_special_tokens=True,
            truncation=True,
            max_length=max_tokens + 16,
        )
        if len(chk) <= max_tokens:
            return s
        n -= 1
    ids = tokenizer.encode(
        text, add_special_tokens=True, truncation=True, max_length=16
    )
    return tokenizer.decode(
        ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )


# ---------------------------
# Tính BERTScore
# ---------------------------
def compute_bertscore(candidate: str, reference: str,
                      scorer: BERTScorer,
                      tokenizer: AutoTokenizer,
                      prefix: str = "") -> dict:
    cand_trunc = safe_truncate(candidate, tokenizer)
    ref_trunc  = safe_truncate(reference, tokenizer)

    # Đảm bảo không rỗng sau truncate
    if not cand_trunc.strip():
        cand_trunc = candidate[:100]
    if not ref_trunc.strip():
        ref_trunc  = reference[:100]

    P, R, F1 = scorer.score([cand_trunc], [ref_trunc])

    key = f"bertscore_{prefix}" if prefix else "bertscore"
    return {
        f"{key}_precision" : P.item(),
        f"{key}_recall"    : R.item(),
        f"{key}_f1"        : F1.item(),
    }


# ---------------------------
# Hàm tổng hợp
# ROUGE: tóm tắt tham chiếu vs tóm tắt model
# BERTScore: tóm tắt model vs văn bản gốc (content)
# ---------------------------
def evaluate_summary(candidate: str, reference: str, content: str,
                     scorer: BERTScorer,
                     tokenizer: AutoTokenizer) -> dict:
    results = compute_rouge(preprocess_vi(candidate), preprocess_vi(reference))
    results.update(
        compute_bertscore(candidate, content, scorer, tokenizer, prefix="")
    )
    return results


# =========================
# LOAD MODEL & TOKENIZER MỘT LẦN
# =========================
PHOBERT_NAME = "vinai/phobert-base"

print("[INFO] Đang tải PhoBERT tokenizer...")
phobert_tokenizer = AutoTokenizer.from_pretrained(PHOBERT_NAME, use_fast=False)

print("[INFO] Đang tải BERTScorer...")
bert_scorer = BERTScorer(
    model_type=PHOBERT_NAME,
    num_layers=12,
    lang="vi",
    device="cpu",
    batch_size=8,
)
print("[INFO] Tải xong.\n")


# =========================
# BATCH EVALUATION
# =========================
data = pd.read_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\data_test_main_summodel2.xlsx"
)

df = data[:50].copy()

REF_COL     = "summary_pre"
CAND_COL    = "summary_model"
CONTENT_COL = "content"

metric_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rouge2_precision", "rouge2_recall", "rouge2_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "bertscore_precision", "bertscore_recall", "bertscore_f1",
]
for col in metric_cols:
    if col not in df.columns:
        df[col] = np.nan

for idx, row in df.iterrows():
    candidate = row.get(CAND_COL,    "")
    reference = row.get(REF_COL,     "")
    content   = row.get(CONTENT_COL, "")

    if not all(isinstance(x, str) for x in [candidate, reference, content]):
        continue
    if any(x.strip() == "" for x in [candidate, reference, content]):
        continue

    try:
        scores = evaluate_summary(candidate, reference, content,
                                  bert_scorer, phobert_tokenizer)
        for k, v in scores.items():
            df.at[idx, k] = float(v)
        if idx % 10 == 0:
            print(f"[{idx}/{len(df)}] done")
    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue

df.to_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\data_test_main_summodel2_score.xlsx",
    index=False
)

print("\n=== Điểm trung bình ===")
print(df[metric_cols].mean().round(4).to_string())

In [2]:
from rouge_score import rouge_scorer
from bert_score import BERTScorer
from underthesea import word_tokenize
from transformers import AutoTokenizer
import torch
import pandas as pd
import numpy as np

# ---------------------------
# Tiền xử lý — chỉ dùng cho ROUGE
# ---------------------------
def preprocess_vi(text: str) -> str:
    return word_tokenize(text.strip().replace("\n", " "), format="text")


# ---------------------------
# Tính ROUGE
# ---------------------------
def compute_rouge(candidate: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=False
    )
    scores = scorer.score(reference, candidate)
    return {
        "rouge1_precision" : scores['rouge1'].precision,
        "rouge1_recall"    : scores['rouge1'].recall,
        "rouge1_f1"        : scores['rouge1'].fmeasure,
        "rouge2_precision" : scores['rouge2'].precision,
        "rouge2_recall"    : scores['rouge2'].recall,
        "rouge2_f1"        : scores['rouge2'].fmeasure,
        "rougeL_precision" : scores['rougeL'].precision,
        "rougeL_recall"    : scores['rougeL'].recall,
        "rougeL_f1"        : scores['rougeL'].fmeasure,
    }


# ---------------------------
# Truncate an toàn — PhoBERT: max_position_embeddings = 258
# model_max_length của tokenizer rất lớn → BERTScore không tự cắt đủ.
# decode(encode(...)) có thể dài hơn encode(...) → phải kiểm tra round-trip.
# ---------------------------
def safe_truncate(text: str, tokenizer: AutoTokenizer,
                  max_tokens: int = 250) -> str:
    text = (text or "").strip()
    if not text:
        return ""
    n = max_tokens
    while n >= 16:
        ids = tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=True,
            max_length=n,
        )
        s = tokenizer.decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        chk = tokenizer.encode(
            s,
            add_special_tokens=True,
            truncation=True,
            max_length=max_tokens + 16,
        )
        if len(chk) <= max_tokens:
            return s
        n -= 1
    ids = tokenizer.encode(
        text, add_special_tokens=True, truncation=True, max_length=16
    )
    return tokenizer.decode(
        ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )


# ---------------------------
# Tính BERTScore
# ---------------------------
def compute_bertscore(candidate: str, reference: str,
                      scorer: BERTScorer,
                      tokenizer: AutoTokenizer,
                      prefix: str = "") -> dict:
    cand_trunc = safe_truncate(candidate, tokenizer)
    ref_trunc  = safe_truncate(reference, tokenizer)

    # Đảm bảo không rỗng sau truncate
    if not cand_trunc.strip():
        cand_trunc = candidate[:100]
    if not ref_trunc.strip():
        ref_trunc  = reference[:100]

    P, R, F1 = scorer.score([cand_trunc], [ref_trunc])

    key = f"bertscore_{prefix}" if prefix else "bertscore"
    return {
        f"{key}_precision" : P.item(),
        f"{key}_recall"    : R.item(),
        f"{key}_f1"        : F1.item(),
    }


# ---------------------------
# Hàm tổng hợp
# ROUGE: tóm tắt tham chiếu vs tóm tắt model
# BERTScore: tóm tắt model vs văn bản gốc (content)
# ---------------------------
def evaluate_summary(candidate: str, reference: str, content: str,
                     scorer: BERTScorer,
                     tokenizer: AutoTokenizer) -> dict:
    results = compute_rouge(preprocess_vi(candidate), preprocess_vi(reference))
    results.update(
        compute_bertscore(candidate, content, scorer, tokenizer, prefix="")
    )
    return results


# =========================
# LOAD MODEL & TOKENIZER MỘT LẦN
# =========================
PHOBERT_NAME = "vinai/phobert-base"

print("[INFO] Đang tải PhoBERT tokenizer...")
phobert_tokenizer = AutoTokenizer.from_pretrained(PHOBERT_NAME, use_fast=False)

print("[INFO] Đang tải BERTScorer...")
bert_scorer = BERTScorer(
    model_type=PHOBERT_NAME,
    num_layers=12,
    lang="vi",
    device="cpu",
    batch_size=8,
)
print("[INFO] Tải xong.\n")


# =========================
# BATCH EVALUATION
# =========================
data = pd.read_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_test_50_sum.xlsx"
)

df = data[:50].copy()

REF_COL     = "summary_pre"
CAND_COL    = "summary_model"
CONTENT_COL = "content"

metric_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rouge2_precision", "rouge2_recall", "rouge2_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "bertscore_precision", "bertscore_recall", "bertscore_f1",
]
for col in metric_cols:
    if col not in df.columns:
        df[col] = np.nan

for idx, row in df.iterrows():
    candidate = row.get(CAND_COL,    "")
    reference = row.get(REF_COL,     "")
    content   = row.get(CONTENT_COL, "")

    if not all(isinstance(x, str) for x in [candidate, reference, content]):
        continue
    if any(x.strip() == "" for x in [candidate, reference, content]):
        continue

    try:
        scores = evaluate_summary(candidate, reference, content,
                                  bert_scorer, phobert_tokenizer)
        for k, v in scores.items():
            df.at[idx, k] = float(v)
        if idx % 10 == 0:
            print(f"[{idx}/{len(df)}] done")
    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue

df.to_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_test_50_sum_score.xlsx",
    index=False
)

print("\n=== Điểm trung bình ===")
print(df[metric_cols].mean().round(4).to_string())

[INFO] Đang tải PhoBERT tokenizer...
[INFO] Đang tải BERTScorer...
[INFO] Tải xong.

[0/50] done
[10/50] done
[20/50] done
[30/50] done
[40/50] done

=== Điểm trung bình ===
rouge1_precision       0.9115
rouge1_recall          0.6003
rouge1_f1              0.7081
rouge2_precision       0.7434
rouge2_recall          0.4918
rouge2_f1              0.5795
rougeL_precision       0.7663
rougeL_recall          0.5068
rougeL_f1              0.5970
bertscore_precision    0.9279
bertscore_recall       0.7222
bertscore_f1           0.8092
